# Fine Tuning de Imagenes con Keras y FastAI

El fine tuning de una red neuronal consiste en ajustar un modelo preentrenado con un conjunto de datos mas pequeno y especifico para nuestra tarea.

En este notebook vamos a entrenar un clasificador de imagenes (bananas maduras vs normales) usando las librerias de [Keras](https://keras.io/) y [FastAI](https://docs.fast.ai/) que son las que me permiten hacer esto de la manera mas simple. Si queremos mas control podemos optar por usar directamente PyTorch.

## Descargar y preparar el dataset de bananas

Usaremos imagenes de ejemplo del curso y construiremos un directorio `images` con dos carpetas:
- `normal`: donde estaran las imagenes de la clase normal
- `madura`: donde estaran las imagenes de la clase madura

Esto es el formato estandar para clasificacion de imagenes: una carpeta para cada clase.

In [ ]:
from pathlib import Path
import requests
import zipfile

base_url = "https://github.com/amiune/freecodingtour/raw/main/cursos/espanol/deeplearning/data/bananas/images.zip"

download_path = Path("images.zip")
response = requests.get(base_url)
Path("images.zip").write_bytes(response.content)

with zipfile.ZipFile(download_path, 'r') as zip_ref:
        zip_ref.extractall(Path("."))

## FastAI

In [ ]:
from fastai.vision.all import *

path = Path("images")
fnames = get_image_files(path)

dls = ImageDataLoaders.from_path_func(path, fnames, label_func=parent_label, item_tfms=Resize(224), batch_tfms=aug_transforms(), bs=8)
learn = vision_learner(dls, resnet34, metrics=accuracy)

learn.fine_tune(epochs=5)

learn.predict("banana.jpg")

## Keras

Instalaremos Keras si hace falta.

```bash
!pip install keras-cv -U -q
!pip install keras-hub -U -q
!pip install keras -U -q
```

In [ ]:
import keras_cv
import keras
import numpy as np

model = keras_cv.models.ImageClassifier.from_preset(
    "resnet50_imagenet",
    num_classes=2 # Added num_classes based on the dataset
)

train_ds = keras.utils.image_dataset_from_directory("images")

# Compile the model with an optimizer, loss function, and metrics
model.compile(
    optimizer=keras.optimizers.Adam(),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)

model.fit(train_ds)

# Load and preprocess the image for prediction
img = keras.utils.load_img("banana.jpg", target_size=(224, 224)) # ResNet50 input size
img_array = keras.utils.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension

predictions = model.predict(img_array)
print("Predictions for banana.jpg:", predictions)

# Determine and print the predicted class
predicted_class_index = np.argmax(predictions)
class_names = train_ds.class_names
predicted_class_name = class_names[predicted_class_index]
print(f"The predicted class for banana.jpg is: {predicted_class_name}")

## Referencias

- [Keras](https://keras.io/examples/)
- [FastAI](https://docs.fast.ai/)

# Fin: [Volver al contenido del curso](https://www.freecodingtour.com/cursos/espanol/deeplearning/deeplearning.html)